In [ ]:
# ============================
# doi_pdf_downloader.py
# ============================
# Drop-in, testable Python code split into small parts.
# 1) Put your DOIs into the DOIS list below (URLs or bare DOIs both fine).
# 2) Set UNPAYWALL_EMAIL.
# 3) Run:  python doi_pdf_downloader.py
#
# What it does:
# - Resolves DOI -> Open Access PDF via Unpaywall (preferred) or Crossref (fallback).
# - Verifies the URL is a real PDF (Content-Type or %PDF header).
# - Saves files into ./pdfs and writes ./pdfs/download_summary.json

import os
import re
import json
import time
import pathlib
import argparse
from typing import Optional, Tuple, List, Dict
import requests

In [ ]:
# Cell 1: Config & DOIs
import os, re, json, time, pathlib
from typing import Optional, Tuple, List, Dict
import requests

UNPAYWALL_EMAIL = os.environ.get("UNPAYWALL_EMAIL", "you@your.org")  # <- set or export env var
OUTPUT_DIR = pathlib.Path(r"C:\Users\israe\OneDrive\Documents\students\sonya\papers")
POLITE_DELAY_SEC = 0.4

DOIS: List[str] = [
    "https://doi.org/10.1016/j.cortex.2020.12.023",
    "https://doi.org/10.15842/KJCP.2016.35.3.007",
    "https://doi.org/10.1016/j.schres.2020.10.005",
    "https://doi.org/10.1037/emo0000404",
    "https://doi.org/10.1016/j.scog.2022.100253",
    "https://doi.org/10.1038/s41598-023-35773-6",
    "https://doi.org/10.1002/aur.2781",
    "https://doi.org/10.1016/j.cortex.2019.01.024",
    "https://doi.org/10.1093/scan/nsy102",
    "https://doi.org/10.1167/jov.21.9.2530",
    "https://doi.org/10.1177/1087054718808602",
    "https://doi.org/10.1007/s10803-015-2630-3",
    "https://doi.org/10.1016/j.psychres.2012.03.004",
    "https://doi.org/10.1016/j.psychres.2012.07.007",
    "https://doi.org/10.31234/osf.io/e7w6k",
    "https://doi.org/10.1044/2024_JSLHR-23-00373",
    "https://doi.org/10.1057/s41599-020-0499-z",
    "https://doi.org/10.1080/02699052.2024.2311336",
    "https://doi.org/10.1016/j.cognition.2020.104345",
    "https://doi.org/10.3389/fcomm.2024.1366597",
    "https://doi.org/10.1016/j.psychres.2016.05.037",
    "https://doi.org/10.1016/j.psychres.2018.03.044",
    "https://doi.org/10.1037/a0023746",
]







len(DOIS)


In [ ]:
# Cell 2: Helpers
PDF_CT = "application/pdf"
PDF_MAGIC = b"%PDF"

HEADERS = {
    "User-Agent": "doi-to-pdf/1.2 (+https://example.org)",
    "Accept": "*/*",
    "Accept-Language": "en",
}

def is_doi(s: str) -> bool:
    return bool(re.match(r"^10\.\d{4,9}/\S+$", s.strip(), re.I))

def clean_doi(s: str) -> str:
    s = s.strip()
    s = re.sub(r"^https?://(dx\.)?doi\.org/", "", s, flags=re.I)
    return s

def sanitize_filename_from_doi(doi: str) -> str:
    return re.sub(r"[^\w.-]+", "_", doi) + ".pdf"

def looks_like_pdf_response(resp: requests.Response, first_chunk: bytes) -> bool:
    ct = (resp.headers.get("Content-Type") or "").split(";")[0].strip().lower()
    if ct == PDF_CT:
        return True
    return first_chunk.startswith(PDF_MAGIC)

def prepare_doi_list(input_items: List[str]) -> List[str]:
    cleaned = []
    for s in input_items:
        s = clean_doi(s)
        if is_doi(s):
            cleaned.append(s)
    return sorted(set(cleaned))

prepared = prepare_doi_list(DOIS)
len(prepared), prepared[:5]


In [ ]:
# Cell 3: Unpaywall resolver
UNPAYWALL_API = "https://api.unpaywall.org/v2/"

def get_unpaywall_pdf_url(doi: str, email: str) -> Optional[str]:
    if not email:
        return None
    url = f"{UNPAYWALL_API}{requests.utils.quote(doi)}"
    r = requests.get(url, params={"email": email}, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return None
    data = r.json()
    for key in ("best_oa_location", "oa_location"):
        loc = data.get(key) or {}
        pdf = loc.get("url_for_pdf") or loc.get("url")
        if pdf:
            return pdf
    for loc in data.get("oa_locations", []) or []:
        pdf = loc.get("url_for_pdf") or loc.get("url")
        if pdf:
            return pdf
    return None

# Quick smoke test on first 3 DOIs (just resolves URL; no download yet)
test_urls_unpaywall = {d: get_unpaywall_pdf_url(d, UNPAYWALL_EMAIL) for d in prepared[:3]}
test_urls_unpaywall


In [ ]:
# Cell 4: Crossref resolver
CROSSREF_API = "https://api.crossref.org/works/"

def get_crossref_pdf_url(doi: str) -> Optional[str]:
    url = f"{CROSSREF_API}{requests.utils.quote(doi)}"
    r = requests.get(url, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return None
    data = r.json()
    item = data.get("message", {})
    for link in item.get("link", []) or []:
        if (link.get("content-type") or "").lower() == PDF_CT:
            return link.get("URL")
    return None

# Smoke test fallback for first 3 DOIs
test_urls_crossref = {d: get_crossref_pdf_url(d) for d in prepared[:3]}
test_urls_crossref


In [ ]:
# Cell 5: Resolve strategy + one-off tester
def resolve_pdf_url(doi: str, email: str) -> Tuple[Optional[str], str]:
    pdf = get_unpaywall_pdf_url(doi, email)
    if pdf:
        return pdf, "unpaywall"
    pdf = get_crossref_pdf_url(doi)
    if pdf:
        return pdf, "crossref"
    return None, "none"

# Try just ONE DOI end-to-end resolution (no download)
one = prepared[0]
resolve_pdf_url(one, UNPAYWALL_EMAIL)


In [ ]:
# Cell 6: Downloader (streams & verifies PDF)
def download_pdf(pdf_url: str, outpath: pathlib.Path) -> bool:
    with requests.get(
        pdf_url,
        headers={**HEADERS, "Accept": PDF_CT},
        timeout=30,
        stream=True,
        allow_redirects=True,
    ) as r:
        r.raise_for_status()
        it = r.iter_content(chunk_size=65536)
        try:
            first = next(it)
        except StopIteration:
            return False
        if not looks_like_pdf_response(r, first):
            return False
        outpath.parent.mkdir(parents=True, exist_ok=True)
        with open(outpath, "wb") as f:
            f.write(first)
            for chunk in it:
                if chunk:
                    f.write(chunk)
    return True

# Try a single DOI full path (resolve + download) to check permissions/network
test_doi = prepared[0]
pdf_url, source = resolve_pdf_url(test_doi, UNPAYWALL_EMAIL)
pdf_url, source


In [ ]:
# Cell 7: Process first 3 DOIs only
def process_dois(dois: List[str], outdir: pathlib.Path, email: str, delay: float = 0.4) -> List[Dict]:
    results = []
    for doi in dois:
        print(f"→ {doi}")
        try:
            pdf_url, source = resolve_pdf_url(doi, email)
            if not pdf_url:
                print("   No OA PDF found (paywalled or unavailable).")
                results.append({"doi": doi, "status": "no_pdf", "source": source, "url": None})
                time.sleep(delay); continue
            outname = sanitize_filename_from_doi(doi)
            outpath = outdir / outname
            ok = download_pdf(pdf_url, outpath)
            if ok:
                print(f"   Downloaded from {source}: {outname}")
                results.append({"doi": doi, "status": "downloaded", "source": source, "url": pdf_url, "file": str(outpath)})
            else:
                print(f"   Found URL but it is not a PDF (or blocked): {pdf_url}")
                results.append({"doi": doi, "status": "not_pdf", "source": source, "url": pdf_url})
        except requests.HTTPError as e:
            print(f"   HTTP error: {e}")
            results.append({"doi": doi, "status": "http_error", "error": str(e)})
        except Exception as e:
            print(f"   Error: {e}")
            results.append({"doi": doi, "status": "error", "error": str(e)})
        time.sleep(delay)
    return results

subset_results = process_dois(prepared[:3], OUTPUT_DIR, UNPAYWALL_EMAIL, delay=POLITE_DELAY_SEC)
subset_results


In [ ]:
# Cell 8: Full run — all 40 DOIs
def write_summary(results: List[Dict], outdir: pathlib.Path) -> pathlib.Path:
    summary_path = outdir / "download_summary.json"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
    return summary_path

all_results = process_dois(prepared, OUTPUT_DIR, UNPAYWALL_EMAIL, delay=POLITE_DELAY_SEC)
summary_path = write_summary(all_results, OUTPUT_DIR)
summary_path, len(all_results)
